# Grau de Severidade de restritivos — GS1 a GS5Portfólio de modelos · Estudo 03. Notebook completo: gera a base, constrói as variáveis,segmenta, valida contra um alvo externo e mede a suavização.**Semente fixa `20260906`.** As células estão na mesma ordem dos scripts que geraram orelatório, e o código foi recortado deles — não redigitado. Rodando de cima a baixo, osnúmeros batem com os da página.**A base é sintética.** Nenhum dado de pessoa real. As proporções foram calibradas contraestatísticas públicas (Serasa, IBGE) para que a base tenha o formato do Brasil negativado.Tempo total de execução: cerca de dois minutos.

---## 1. A baseCinco arquétipos latentes de comportamento geram as dívidas. **A segmentação nunca vê essesrótulos** — eles servem só para, no fim, medir quanto da estrutura o algoritmo redescobriu.A prescrição de 60 meses da Lei 12.414/2011 está no gerador: passado esse prazo o registrocai, com ou sem pagamento. É como o devedor crônico sai da base.

In [ ]:
import numpy as npimport pandas as pdSEMENTE = 20260906rng = np.random.default_rng(SEMENTE)N_PESSOAS = 300_000DATA_REF = pd.Timestamp("2026-07-31")JANELA_M = 84  # 7 anos de historico: alem dos 60 meses de permanencia legal,               # para que a prescricao apareca como forma de saida# ---------------------------------------------------------------- macrogrupos# share de CONTAGEM de dividas. G5 deliberadamente baixo: financiamento com# garantia costuma virar retomada do bem antes de virar negativacao.MACRO = {    1: dict(nome="Essencial e utilities",      p=0.367, mediana=  310, sigma=0.85),    2: dict(nome="Servico recorrente",         p=0.305, mediana=  390, sigma=0.90),    3: dict(nome="Varejo e parcelado",         p=0.200, mediana=  610, sigma=1.05),    4: dict(nome="Credito bancario sem garantia", p=0.258, mediana=1850, sigma=1.25),    5: dict(nome="Credito com garantia",       p=0.021, mediana=14000, sigma=1.10),    6: dict(nome="Judicial e fiscal",          p=0.043, mediana= 3900, sigma=1.15),}# ---------------------------------------------------------------- arquetipos# p        share da populacao# n_div    media de dividas (Poisson deslocada)# mix      peso relativo por macrogrupo (multiplica o share base)# comp     alvo de comprometimento (valor total / renda mensal), lognormal# meses    dispersao temporal das inclusoes (menor = mais concentrado)# taxa_bx  propensao a dar baixa no restritivo# escalada tendencia de inclusoes no semestre recente vs o anterior# o90      probabilidade de over90 mob3 no publico interno# p_banco  probabilidade de ser cliente do banco (vies de selecao deliberado)ARQ = {    "Esquecimento": dict(        p=0.30, n_div=1.15, mix=[3.2, 2.4, 0.7, 0.25, 0.03, 0.10],        comp=(0.045, 0.75), meses=0.35, taxa_bx=0.78, escalada=0.92,        o90=0.030, p_banco=0.52),    "Aperto pontual": dict(        p=0.22, n_div=2.70, mix=[2.1, 1.5, 1.5, 0.65, 0.10, 0.25],        comp=(0.16, 0.70), meses=0.55, taxa_bx=0.55, escalada=1.00,        o90=0.072, p_banco=0.44),    "Consumo acima da renda": dict(        p=0.20, n_div=4.70, mix=[0.9, 1.0, 2.3, 1.7, 0.25, 0.45],        comp=(0.52, 0.72), meses=0.75, taxa_bx=0.34, escalada=1.12,        o90=0.150, p_banco=0.33),    "Credito estourado": dict(        p=0.18, n_div=7.15, mix=[0.8, 0.8, 1.4, 2.9, 0.55, 0.90],        comp=(1.35, 0.78), meses=0.85, taxa_bx=0.19, escalada=1.30,        o90=0.305, p_banco=0.22),    "Cronico e judicial": dict(        p=0.10, n_div=11.10, mix=[1.1, 0.9, 1.2, 2.2, 1.30, 3.40],        comp=(3.10, 0.85), meses=1.00, taxa_bx=0.08, escalada=1.06,        o90=0.520, p_banco=0.13),}NOMES_ARQ = list(ARQ)P_ARQ = np.array([ARQ[a]["p"] for a in NOMES_ARQ])def gerar_pessoas():    arq = rng.choice(len(NOMES_ARQ), size=N_PESSOAS, p=P_ARQ)    # idade conforme a distribuicao Serasa de jul/2026    faixas = rng.choice(4, size=N_PESSOAS, p=np.array([0.110,0.332,0.332,0.201])/0.975)    lim = {0: (18, 25), 1: (26, 40), 2: (41, 60), 3: (61, 82)}    idade = np.array([rng.integers(*lim[f], endpoint=True) for f in faixas])    # renda: lognormal calibrada para media ~R$ 3.722 (PNAD 1T2026).    # A populacao negativada e mais pobre que a media do pais, entao o alvo    # aqui e um pouco abaixo: aplicamos o desconto no fim.    sigma_r = 0.72    mu_r = np.log(2950) - 0.0    renda = rng.lognormal(mu_r, sigma_r, N_PESSOAS)    # jovens e idosos ganham menos    renda *= np.where(faixas == 0, 0.62, np.where(faixas == 3, 0.84, 1.0))    renda = np.clip(renda, 700, 90_000)    cli = pd.DataFrame({        "id_pessoa": np.arange(1, N_PESSOAS + 1),        "arquetipo": [NOMES_ARQ[a] for a in arq],        "idade": idade,        "renda_mensal": renda.round(2),    })    # vies de selecao: o banco aprovou mais gente dos arquetipos leves    pb = cli.arquetipo.map(lambda a: ARQ[a]["p_banco"]).to_numpy()    cli["cliente_banco"] = rng.random(N_PESSOAS) < pb    return clidef gerar_restritivos(cli):    linhas = []    base_share = np.array([MACRO[g]["p"] for g in range(1, 7)])    for nome in NOMES_ARQ:        a = ARQ[nome]        idx = np.flatnonzero((cli.arquetipo == nome).to_numpy())        n = len(idx)        # quantidade de restritivos: Poisson + 1 (todo negativado tem ao menos um)        qtd = rng.poisson(a["n_div"], n) + 1        total = int(qtd.sum())        # macrogrupo: share base ponderado pelo mix do arquetipo        peso = base_share * np.array(a["mix"])        peso = peso / peso.sum()        grupo = rng.choice(np.arange(1, 7), size=total, p=peso)        # valor: lognormal por macrogrupo        med = np.array([MACRO[g]["mediana"] for g in grupo])        sig = np.array([MACRO[g]["sigma"] for g in grupo])        valor = rng.lognormal(np.log(med), sig)        # data de inclusao: exponencial truncada, mais densa perto da referencia        conc = a["meses"]        u = rng.random(total)        idade_m = np.clip((-np.log(1 - u * 0.985) / (0.055 + 0.10 * (1 - conc))),                          0, JANELA_M - 1)        # escalada: empurra parte das inclusoes para os 6 meses recentes        if a["escalada"] > 1:            mover = rng.random(total) < (a["escalada"] - 1) * 0.75            idade_m = np.where(mover, rng.random(total) * 6, idade_m)        pessoa = np.repeat(cli.id_pessoa.to_numpy()[idx], qtd)        # baixa: mais provavel em divida antiga e de valor baixo        prop = a["taxa_bx"] * (0.45 + 0.55 * np.clip(idade_m / 24, 0, 1))        prop *= np.clip(1.35 - 0.30 * np.log10(np.maximum(valor, 50) / 300), 0.25, 1.5)        baixado = rng.random(total) < np.clip(prop, 0, 0.97)        # a baixa ocorre entre a inclusao e a data de referencia        idade_bx = np.where(baixado, rng.random(total) * idade_m, np.nan)        # prescricao: 60 meses apos a inclusao o registro cai por lei,        # com ou sem pagamento        prescrito = (~baixado) & (idade_m >= 60)        linhas.append(pd.DataFrame({            "id_pessoa": pessoa,            "macrogrupo": grupo,            "valor": valor,            "meses_desde_inclusao": idade_m,            "baixado": baixado,            "prescrito": prescrito,            "meses_desde_baixa": idade_bx,        }))    r = pd.concat(linhas, ignore_index=True)    # ---- calibragem de valor: media por pessoa = R$ 6.598,13 (Serasa fev/2026)    # so as dividas ATIVAS contam para o estoque negativado divulgado    r["ativo"] = (~r.baixado) & (~r.prescrito)    # meses desde que o registro saiu da base, por qualquer das duas vias    r["meses_desde_saida"] = np.where(        r.baixado, r.meses_desde_baixa,        np.where(r.prescrito, r.meses_desde_inclusao - 60, np.nan))    ativo = r[r.ativo]    media_obs = ativo.groupby("id_pessoa").valor.sum().mean()    fator = 6598.13 / media_obs    r["valor"] = (r.valor * fator).round(2)    r["dt_inclusao"] = DATA_REF - pd.to_timedelta(r.meses_desde_inclusao * 30.44, unit="D")    r["dt_baixa"] = DATA_REF - pd.to_timedelta(r.meses_desde_baixa * 30.44, unit="D")    r["macrogrupo_nome"] = r.macrogrupo.map(lambda g: MACRO[g]["nome"])    r.insert(0, "id_restritivo", np.arange(1, len(r) + 1))    return r, fatordef gerar_comportamento_interno(cli):    """over90 em mob3, mob6 e mob12 para quem e cliente do banco.    A probabilidade vem do ARQUETIPO LATENTE, nunca do grupo que a    segmentacao vai encontrar. E o que torna a validacao externa honesta.    """    m = cli[cli.cliente_banco].copy()    p3 = m.arquetipo.map(lambda a: ARQ[a]["o90"]).to_numpy()    # dentro do arquetipo, renda baixa piora um pouco    ajuste = np.clip((3000 / np.maximum(m.renda_mensal.to_numpy(), 500)) ** 0.28, 0.72, 1.55)    p3 = np.clip(p3 * ajuste, 0.002, 0.93)    o3 = rng.random(len(m)) < p3    # mob6 e mob12 acumulam: quem ja estourou continua estourado    p6 = np.clip(p3 * 1.55, 0, 0.95)    p12 = np.clip(p3 * 2.05, 0, 0.97)    o6 = o3 | (rng.random(len(m)) < (p6 - p3) / np.maximum(1 - p3, 1e-6))    o12 = o6 | (rng.random(len(m)) < (p12 - p6) / np.maximum(1 - p6, 1e-6))    return pd.DataFrame({        "id_pessoa": m.id_pessoa.to_numpy(),        "over90_mob3": o3, "over90_mob6": o6, "over90_mob12": o12,    })

In [ ]:
cli = gerar_pessoas()res, fator = gerar_restritivos(cli)comp = gerar_comportamento_interno(cli)cli.to_csv("base_pessoas.csv", index=False)res.drop(columns=["meses_desde_baixa"]).to_csv("base_restritivos.csv", index=False)comp.to_csv("base_comportamento.csv", index=False)ativo = res[res.ativo]print(f"pessoas ................. {len(cli):,}".replace(",", "."))print(f"restritivos (total) ..... {len(res):,}".replace(",", "."))print(f"restritivos ativos ...... {len(ativo):,}".replace(",", "."))print(f"baixados por pagamento .. {int(res.baixado.sum()):,}".replace(",", ".")      + f"   prescritos: {int(res.prescrito.sum()):,}".replace(",", "."))print(f"ativos por pessoa ....... {len(ativo)/len(cli):.2f}   (alvo 4,06)")print(f"divida ativa media ...... R$ {ativo.groupby('id_pessoa').valor.sum().mean():,.2f}"      .replace(",", "X").replace(".", ",").replace("X", "."), " (alvo R$ 6.598,13)")print(f"fator de calibragem ..... {fator:.4f}")print(f"renda media ............. R$ {cli.renda_mensal.mean():,.0f}"      .replace(",", "."), " (PNAD R$ 3.722)")print(f"clientes do banco ....... {cli.cliente_banco.mean():.1%}  n={cli.cliente_banco.sum():,}"      .replace(",", "."))print(f"over90 mob3 (interno) ... {comp.over90_mob3.mean():.2%}")print()print("composicao por macrogrupo (contagem, so ativos):")cs = ativo.macrogrupo_nome.value_counts(normalize=True)for k, v in cs.items():    print(f"  {k:32s} {v:6.1%}")print()print("faixa etaria:")fx = pd.cut(cli.idade, [17, 25, 40, 60, 120],            labels=["18-25", "26-40", "41-60", "60+"])for k, v in fx.value_counts(normalize=True).sort_index().items():    print(f"  {k:6s} {v:6.1%}")

---## 2. As variáveis, em cinco blocosSeveridade financeira, natureza, volume, tempo e saída. O alvo interno (`over90`) **nãoentra aqui** — ele é validação, não insumo.O bloco de natureza é onde a camada 1 acontece: seis categorias nominais viram um índiceordinal ponderado por valor, o que permite a natureza entrar numa distância euclidiana semvirar dummy.

In [ ]:
import numpy as npimport pandas as pd# Severidade ordinal do macrogrupo. E o que resolve o problema de colocar# categoria nominal dentro de uma distancia euclidiana: em vez de seis dummies# sem ordem, uma escala com significado de negocio.SEVERIDADE = {1: 1.0, 2: 1.0, 3: 2.0, 4: 3.0, 5: 4.0, 6: 5.0}BLOCOS = {    "A. Severidade financeira": ["comprometimento", "maior_sobre_renda", "concentracao"],    "B. Natureza":              ["indice_natureza", "n_macrogrupos", "share_bancario", "tem_judicial"],    "C. Volume":                ["qtd_ativos"],    "D. Tempo":                 ["idade_mais_antigo", "recencia", "meses_com_inclusao_12m", "escalada"],    "E. Saida":                 ["qtd_baixas_12m", "razao_baixa"],}# eixos que entram na clusterizacao (camada 2)EIXOS = ["comprometimento", "concentracao", "indice_natureza",         "qtd_ativos", "meses_com_inclusao_12m", "escalada", "razao_baixa"]ROTULOS = {    "comprometimento": "Valor ativo ÷ renda mensal",    "maior_sobre_renda": "Maior restritivo ÷ renda",    "concentracao": "Maior restritivo ÷ valor total",    "indice_natureza": "Índice de natureza (1 a 5)",    "n_macrogrupos": "Macrogrupos distintos",    "share_bancario": "% do valor em crédito bancário",    "tem_judicial": "Tem restritivo judicial ou fiscal",    "qtd_ativos": "Restritivos ativos",    "idade_mais_antigo": "Idade do mais antigo (meses)",    "recencia": "Meses desde a inclusão mais recente",    "meses_com_inclusao_12m": "Meses distintos com inclusão em 12M",    "escalada": "Inclusões 6M recentes ÷ 6M anteriores",    "qtd_baixas_12m": "Baixas nos últimos 12 meses",    "razao_baixa": "Baixas ÷ (baixas + inclusões), 12M",}def construir(pes, res):    res = res.copy()    res["sev"] = res.macrogrupo.map(SEVERIDADE)    ativo = res[res.ativo]    g = ativo.groupby("id_pessoa")    f = pd.DataFrame(index=pes.id_pessoa)    # ---- C. volume    f["qtd_ativos"] = g.size()    # ---- A. severidade financeira    total = g.valor.sum()    maior = g.valor.max()    renda = pes.set_index("id_pessoa").renda_mensal    f["valor_total"] = total    f["comprometimento"] = total / renda    f["maior_sobre_renda"] = maior / renda    f["concentracao"] = maior / total    # ---- B. natureza    ativo_ = ativo.assign(vs=ativo.valor * ativo.sev)    f["indice_natureza"] = ativo_.groupby("id_pessoa").vs.sum() / total    f["n_macrogrupos"] = g.macrogrupo.nunique()    banc = ativo[ativo.macrogrupo == 4].groupby("id_pessoa").valor.sum()    f["share_bancario"] = (banc / total).fillna(0.0)    f["tem_judicial"] = (ativo[ativo.macrogrupo == 6].groupby("id_pessoa").size()                         .reindex(f.index).fillna(0) > 0).astype(int)    # ---- D. tempo    f["idade_mais_antigo"] = g.meses_desde_inclusao.max()    f["recencia"] = g.meses_desde_inclusao.min()    m12 = ativo[ativo.meses_desde_inclusao < 12].copy()    m12["mes"] = np.floor(m12.meses_desde_inclusao).astype(int)    f["meses_com_inclusao_12m"] = (m12.groupby("id_pessoa").mes.nunique()                                   .reindex(f.index).fillna(0))    rec = (ativo[ativo.meses_desde_inclusao < 6].groupby("id_pessoa").size()           .reindex(f.index).fillna(0))    ant = (ativo[(ativo.meses_desde_inclusao >= 6) & (ativo.meses_desde_inclusao < 12)]           .groupby("id_pessoa").size().reindex(f.index).fillna(0))    f["escalada"] = (rec + 1) / (ant + 1)    # ---- E. saida    bx = res[res.baixado & (res.meses_desde_saida < 12)]    f["qtd_baixas_12m"] = bx.groupby("id_pessoa").size().reindex(f.index).fillna(0)    incl12 = (ativo[ativo.meses_desde_inclusao < 12].groupby("id_pessoa").size()              .reindex(f.index).fillna(0))    f["razao_baixa"] = f.qtd_baixas_12m / (f.qtd_baixas_12m + incl12 + 1)    f = f.reset_index()    f = f.merge(pes[["id_pessoa", "arquetipo", "idade", "renda_mensal",                     "cliente_banco"]], on="id_pessoa")    # Quem teve TODOS os restritivos baixados nao esta negativado hoje: sai da    # populacao a segmentar. Volta depois, como coorte de ex-negativados, na    # analise de suavizacao.    f["negativado_hoje"] = f.qtd_ativos.notna()    return f

In [ ]:
pes = pd.read_csv("base_pessoas.csv")res = pd.read_csv("base_restritivos.csv")f = construir(pes, res)ex = f[~f.negativado_hoje].copy()f = f[f.negativado_hoje].drop(columns=["negativado_hoje"]).reset_index(drop=True)f.to_csv("features_pessoas.csv", index=False)ex[["id_pessoa", "arquetipo", "idade", "renda_mensal", "cliente_banco"]] \    .to_csv("base_ex_negativados.csv", index=False)print(f"negativados hoje: {len(f):,}".replace(",", ".")      + f"   ex-negativados: {len(ex):,}".replace(",", ".")      + f"   colunas: {f.shape[1]}")print(f"nulos: {int(f.isna().sum().sum())}")print()cols = [c for b in BLOCOS.values() for c in b]d = f[cols].describe().T[["mean", "50%", "std", "min", "max"]]d["skew"] = f[cols].skew()print(d.round(3).to_string())print()print("media por arquetipo (o que a clusterizacao tem de redescobrir):")print(f.groupby("arquetipo")[EIXOS].mean().round(2).to_string())

---## 3. A segmentaçãoTransformação (`log` para razões, `log1p` para contagens), padronização, K-Means com k=5.Repare no diagnóstico de k: as três métricas preferem **k=2**. O modelo usa cinco porquepolítica de crédito precisa de faixas — e é a validação da seção seguinte, não a métricainterna, que legitima essa escolha.

In [ ]:
import jsonimport numpy as npimport pandas as pdfrom sklearn.cluster import KMeansfrom sklearn.decomposition import PCAfrom sklearn.metrics import (silhouette_score, davies_bouldin_score,                             calinski_harabasz_score, adjusted_rand_score)from sklearn.preprocessing import StandardScalerSEMENTE = 20260906K = 5   # fixado pelo desenho do modelo: cinco graus de severidade# log simples para razoes e proporcoes (valores pequenos, cauda multiplicativa);# log1p para contagens, que valem zero.TRANSFORMA = {    "comprometimento": "log",    "escalada": "log",    "qtd_ativos": "log1p",    "meses_com_inclusao_12m": "log1p",}def preparar(f):    X = f[EIXOS].copy()    for c, t in TRANSFORMA.items():        X[c] = np.log(X[c]) if t == "log" else np.log1p(X[c])    return Xdef diagnostico_k(Xs, ks=range(2, 9), amostra=10_000, semente=SEMENTE):    rng = np.random.default_rng(semente)    idx = rng.choice(len(Xs), min(amostra, len(Xs)), replace=False)    Xa = Xs[idx]    linhas = []    for k in ks:        km = KMeans(k, n_init=5, random_state=semente).fit(Xs)        lab = km.predict(Xa)        linhas.append(dict(            k=k,            inercia=km.inertia_,            silhueta=silhouette_score(Xa, lab),            davies_bouldin=davies_bouldin_score(Xa, lab),            calinski=calinski_harabasz_score(Xa, lab),        ))    return pd.DataFrame(linhas)def estabilidade(Xs, k, n=8, frac=0.70, semente=SEMENTE):    """ARI entre pares de reamostragens: o mesmo cliente cai junto do mesmo grupo?"""    rng = np.random.default_rng(semente)    rot = []    base = KMeans(k, n_init=20, random_state=semente).fit(Xs)    for i in range(n):        idx = rng.choice(len(Xs), int(len(Xs) * frac), replace=False)        km = KMeans(k, n_init=5, random_state=semente + i + 1).fit(Xs[idx])        r = np.full(len(Xs), -1)        r[idx] = km.predict(Xs[idx])        rot.append(r)    rg2 = np.random.default_rng(semente + 99)    amo = rg2.choice(len(Xs), min(50_000, len(Xs)), replace=False)    aris = []    for i in range(n):        for j in range(i + 1, n):            a, b = rot[i][amo], rot[j][amo]            m = (a >= 0) & (b >= 0)            aris.append(adjusted_rand_score(a[m], b[m]))    return float(np.mean(aris)), float(np.std(aris)), base

In [ ]:
f = pd.read_csv("features_pessoas.csv")X = preparar(f)esc = StandardScaler().fit(X)Xs = esc.transform(X)diag = diagnostico_k(Xs)diag.to_csv("diag_k.csv", index=False)print("escolha de k (a decisao de negocio fixou 5; a metrica e o contraponto)")print(diag.round(4).to_string(index=False))ari_m, ari_s, km = estabilidade(Xs, K)lab = km.predict(Xs)# ---- ordenar os grupos por severidade, sem olhar o alvo internoC = pd.DataFrame(km.cluster_centers_, columns=EIXOS)sev = (C.comprometimento + C.indice_natureza + C.qtd_ativos       + C.meses_com_inclusao_12m - C.razao_baixa)ordem = sev.sort_values().index.to_numpy()de_para = {int(c): i + 1 for i, c in enumerate(ordem)}f["GS"] = pd.Series(lab).map(de_para).to_numpy()rg3 = np.random.default_rng(SEMENTE + 7)amo_s = rg3.choice(len(Xs), 10_000, replace=False)sil = silhouette_score(Xs[amo_s], lab[amo_s])print(f"\nk={K}  silhueta={sil:.4f}  ARI de estabilidade={ari_m:.4f} (dp {ari_s:.4f})")print(f"ARI contra o arquetipo latente: "      f"{adjusted_rand_score(f.arquetipo, f.GS):.4f}")# ---- projecao PCA so para desenhopca = PCA(2, random_state=SEMENTE).fit(Xs)P = pca.transform(Xs)f["pc1"], f["pc2"] = P[:, 0], P[:, 1]print(f"variancia explicada PC1+PC2: {pca.explained_variance_ratio_.sum():.1%}")f.to_csv("pessoas_com_gs.csv", index=False)perfil = f.groupby("GS").agg(    pessoas=("id_pessoa", "size"),    comprometimento=("comprometimento", "median"),    valor_total=("valor_total", "median"),    renda=("renda_mensal", "median"),    qtd_ativos=("qtd_ativos", "median"),    indice_natureza=("indice_natureza", "mean"),    concentracao=("concentracao", "median"),    meses_incl_12m=("meses_com_inclusao_12m", "mean"),    escalada=("escalada", "median"),    razao_baixa=("razao_baixa", "mean"),    tem_judicial=("tem_judicial", "mean"),    share_bancario=("share_bancario", "mean"),    idade=("idade", "median"),)perfil["share"] = perfil.pessoas / perfil.pessoas.sum()perfil.to_csv("perfil_gs.csv")print("\nperfil por GS")print(perfil.round(3).to_string())print("\ncomposicao de arquetipo dentro de cada GS (%)")ct = pd.crosstab(f.GS, f.arquetipo, normalize="index") * 100print(ct.round(1).to_string())# parametros para reaplicar fora do Python (planilha, SQL)json.dump({    "semente": SEMENTE, "k": K, "eixos": EIXOS,    "transforma": TRANSFORMA,    "media": esc.mean_.tolist(), "escala": esc.scale_.tolist(),    "centroides": {str(de_para[i]): km.cluster_centers_[i].tolist()                   for i in range(K)},    "silhueta": float(sil), "ari_estabilidade": ari_m,    "ari_arquetipo": float(adjusted_rand_score(f.arquetipo, f.GS)),}, open("parametros_gs.json", "w"), indent=1)print("\nparametros_gs.json gravado")

---## 4. A validação externaAqui, e só aqui, o alvo entra. `over90 mob3` é a definição de default alinhada à ResoluçãoCMN 4.966 e a Basileia.Duas coisas para observar na saída: a tabela de viés de seleção, que mostra a política decrédito já tendo filtrado os graus altos; e o teste de GS1 contra GS2, que **não** dádiferença.

In [ ]:
import numpy as npimport pandas as pdfrom scipy import statsfrom sklearn.metrics import roc_auc_scoreALVOS = ["over90_mob3", "over90_mob6", "over90_mob12"]def wilson(k, n, z=1.96):    """IC de proporcao (Wilson): melhor que o normal em taxas baixas."""    if n == 0:        return (np.nan, np.nan)    p = k / n    d = 1 + z**2 / n    c = (p + z**2 / (2 * n)) / d    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d    return (max(0.0, c - h), min(1.0, c + h))def tabela_validacao(m, alvo):    linhas = []    base = m[alvo].mean()    for gs, g in m.groupby("GS"):        n, k = len(g), int(g[alvo].sum())        lo, hi = wilson(k, n)        linhas.append(dict(GS=gs, clientes=n, eventos=k, taxa=k / n,                           ic_baixo=lo, ic_alto=hi, lift=(k / n) / base))    t = pd.DataFrame(linhas)    t["share_clientes"] = t.clientes / t.clientes.sum()    t["share_eventos"] = t.eventos / t.eventos.sum()    return t

In [ ]:
f = pd.read_csv("pessoas_com_gs.csv")comp = pd.read_csv("base_comportamento.csv")m = f.merge(comp, on="id_pessoa", how="inner")print(f"populacao segmentada ........ {len(f):,}".replace(",", "."))print(f"destes, clientes do banco ... {len(m):,}".replace(",", ".")      + f"  ({len(m)/len(f):.1%})")# ---------------------------------------------------- vies de selecaoprint("\n--- vies de selecao: quem virou cliente do banco ---")vs = pd.DataFrame({    "todos_negativados": f.GS.value_counts(normalize=True).sort_index(),    "clientes_do_banco": m.GS.value_counts(normalize=True).sort_index(),})vs["razao"] = vs.clientes_do_banco / vs.todos_negativadosvs.to_csv("vies_selecao.csv")print((vs * 100).round(1).to_string())print("A politica de credito ja filtrou: os graus altos estao sub-representados")print("dentro de casa. A relacao medida aqui nao se transporta para nao-clientes.")# ---------------------------------------------------- validacao por alvoresumo = {}for alvo in ALVOS:    t = tabela_validacao(m, alvo)    t.to_csv(f"validacao_{alvo}.csv", index=False)    auc = roc_auc_score(m[alvo], m.GS)    # monotonicidade: os IC de GS vizinhos se sobrepoem?    mono = bool((t.taxa.diff().dropna() > 0).all())    sobrepoe = [int(t.ic_alto[i]) for i in range(len(t) - 1)                if t.ic_alto[i] > t.ic_baixo[i + 1]]    tau = stats.kendalltau(m.GS, m[alvo]).statistic    resumo[alvo] = dict(base=m[alvo].mean(), auc=auc, gini=2 * auc - 1,                        monotonico=mono, tau=tau,                        lift_max=t.lift.max(), lift_min=t.lift.min(),                        razao=t.taxa.max() / t.taxa.min())    print(f"\n--- {alvo} (base {m[alvo].mean():.2%}) ---")    print(t.assign(        taxa=lambda d: (d.taxa * 100).round(2),        ic_baixo=lambda d: (d.ic_baixo * 100).round(2),        ic_alto=lambda d: (d.ic_alto * 100).round(2),        lift=lambda d: d.lift.round(2),        share_clientes=lambda d: (d.share_clientes * 100).round(1),        share_eventos=lambda d: (d.share_eventos * 100).round(1),    ).to_string(index=False))    print(f"AUC={auc:.4f}  Gini={2*auc-1:.4f}  tau-b={tau:.4f}  "          f"monotonico={mono}  IC sobrepostos entre vizinhos={len(sobrepoe)}")pd.DataFrame(resumo).T.to_csv("resumo_validacao.csv")# ------------------------------------- GS1 x GS2: o bloco de saida vale?print("\n--- GS1 contra GS2: os dois so diferem no bloco de saida ---")a = m[m.GS == 1]["over90_mob3"]b = m[m.GS == 2]["over90_mob3"]tab = np.array([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])chi2, p, _, _ = stats.chi2_contingency(tab)print(f"GS1 razao_baixa alta -> over90 {a.mean():.2%} (n={len(a):,})".replace(",", "."))print(f"GS2 razao_baixa ~0   -> over90 {b.mean():.2%} (n={len(b):,})".replace(",", "."))print(f"razao entre eles: {b.mean()/a.mean():.2f}x   qui-quadrado p={p:.3e}")print("Nos demais eixos os dois grupos sao praticamente identicos:")print(m[m.GS.isin([1, 2])].groupby("GS")[    ["comprometimento", "qtd_ativos", "valor_total", "indice_natureza",     "concentracao", "razao_baixa"]].median().round(3).to_string())

---## 5. O quinto grau paga o próprio custo?Fundir GS1 e GS2 num grau só e comparar a discriminação. Se o AUC não cair, o modelo temuma faixa a mais do que o dado sustenta.

In [ ]:
import numpy as npimport pandas as pdfrom sklearn.cluster import KMeansfrom sklearn.metrics import roc_auc_score, silhouette_scorefrom sklearn.preprocessing import StandardScalerfrom segmentacao import preparar, SEMENTE

In [ ]:
f = pd.read_csv("features_pessoas.csv")comp = pd.read_csv("base_comportamento.csv")X = preparar(f)Xs = StandardScaler().fit_transform(X)rg = np.random.default_rng(SEMENTE + 7)amo = rg.choice(len(Xs), 10_000, replace=False)linhas = []for k in (3, 4, 5, 6):    km = KMeans(k, n_init=20, random_state=SEMENTE).fit(Xs)    lab = km.labels_    C = pd.DataFrame(km.cluster_centers_, columns=EIXOS)    sev = (C.comprometimento + C.indice_natureza + C.qtd_ativos           + C.meses_com_inclusao_12m - C.razao_baixa)    de_para = {int(c): i + 1 for i, c in enumerate(sev.sort_values().index)}    g = pd.Series(lab).map(de_para).to_numpy()    d = f[["id_pessoa"]].assign(G=g).merge(comp, on="id_pessoa")    taxas = d.groupby("G").over90_mob3.mean()    mono = bool((taxas.diff().dropna() > 0).all())    linhas.append(dict(        k=k,        silhueta=silhouette_score(Xs[amo], lab[amo]),        auc_mob3=roc_auc_score(d.over90_mob3, d.G),        auc_mob12=roc_auc_score(d.over90_mob12, d.G),        monotonico=mono,        razao_extremos=taxas.max() / taxas.min(),        menor_grupo=pd.Series(g).value_counts(normalize=True).min(),    ))t = pd.DataFrame(linhas)t.to_csv("sensibilidade_k.csv", index=False)print(t.round(4).to_string(index=False))# o teste direto: fundir GS1 e GS2 do modelo de 5 grausf5 = pd.read_csv("pessoas_com_gs.csv")d5 = f5[["id_pessoa", "GS"]].merge(comp, on="id_pessoa")fundido = d5.GS.replace({2: 1}).map({1: 1, 3: 2, 4: 3, 5: 4})print(f"\nGS de 5 graus            AUC mob3 = {roc_auc_score(d5.over90_mob3, d5.GS):.4f}")print(f"GS com GS1+GS2 fundidos  AUC mob3 = "      f"{roc_auc_score(d5.over90_mob3, fundido):.4f}")print(f"GS de 5 graus            AUC mob12 = "      f"{roc_auc_score(d5.over90_mob12, d5.GS):.4f}")print(f"GS com GS1+GS2 fundidos  AUC mob12 = "      f"{roc_auc_score(d5.over90_mob12, fundido):.4f}")

---## 6. Suavização — quando parar de penalizarCoorte de quem regularizou o principal, controle pareado, diferença por trimestre comintervalo de confiança, e um critério de convergência declarado **antes** de olhar oresultado.O número que sai daqui vale para esta base. O que se transporta para a vida real é oprocedimento.

In [ ]:
import numpy as npimport pandas as pdSEMENTE = 20260906rng = np.random.default_rng(SEMENTE + 31)# meia-vida da recuperacao, em meses, por arquetipo: quem tropecou volta# rapido, quem quebrou volta devagar.TAU = {"Esquecimento": 4.5, "Aperto pontual": 7.0, "Consumo acima da renda": 11.0,       "Credito estourado": 17.0, "Cronico e judicial": 25.0}P_CONTROLE = 0.022      # over90 mob3 de quem nunca foi negativadoN_CONTROLE = 90_000TRIM = 12               # 12 trimestres = 36 meses de acompanhamentodef wilson(k, n, z=1.96):    if n == 0:        return (np.nan, np.nan)    p = k / n    d = 1 + z**2 / n    c = (p + z**2 / (2 * n)) / d    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d    return (max(0.0, c - h), min(1.0, c + h))def ic_diferenca(k1, n1, k2, n2, z=1.96):    """IC da diferenca de duas proporcoes (Wald com correcao de continuidade)."""    p1, p2 = k1 / n1, k2 / n2    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)    d = p1 - p2    return d, d - z * se, d + z * se

In [ ]:
pes = pd.read_csv("base_pessoas.csv")res = pd.read_csv("base_restritivos.csv")fea = pd.read_csv("features_pessoas.csv")[["id_pessoa", "comprometimento"]]# ---- definicao da coorte: quem REGULARIZOU O PRINCIPAL.# "Zerou tudo" seria uma coorte enviesada - so sai quem entrou leve, e o# devedor pesado praticamente nunca zera. A pergunta de politica de credito# e outra: o cliente quitou a divida que pesava e hoje esta leve; a partir# de quando ele pode ser tratado como qualquer outro?maior = res.loc[res.groupby("id_pessoa").valor.idxmax()]coorte = maior[maior.baixado][["id_pessoa", "meses_desde_saida"]]ex = pes.merge(coorte, on="id_pessoa")ex = ex.merge(fea, on="id_pessoa", how="left")ex["comprometimento"] = ex.comprometimento.fillna(0.0)   # sem ativo = zeroex = ex[(ex.cliente_banco)        & (ex.comprometimento < 0.5)                     # hoje esta leve        & (ex.meses_desde_saida < TRIM * 3)].copy()# ---- comportamento do ex-negativado: decai do nivel do arquetipo para o# nivel do controle, com meia-vida propria. Isto e o DADO; a medicao vem# depois e nao conhece TAU.p0 = ex.arquetipo.map(lambda a: ARQ[a]["o90"]).to_numpy()tau = ex.arquetipo.map(TAU).to_numpy()t = ex.meses_desde_saida.to_numpy()p = P_CONTROLE + (p0 - P_CONTROLE) * np.exp(-t / tau)ex["over90_mob3"] = rng.random(len(ex)) < np.clip(p, 0.001, 0.97)# ---- controle pareado: nunca negativados, casados por faixa de renda e idadectrl = pd.DataFrame({    "renda_mensal": rng.choice(ex.renda_mensal.to_numpy(), N_CONTROLE),    "idade": rng.choice(ex.idade.to_numpy(), N_CONTROLE),})# leve gradiente de renda, igual ao aplicado no publico negativadoaj = np.clip((3000 / np.maximum(ctrl.renda_mensal.to_numpy(), 500)) ** 0.28, 0.72, 1.55)ctrl["over90_mob3"] = rng.random(N_CONTROLE) < np.clip(P_CONTROLE * aj, 0.001, 0.9)taxa_ctrl = ctrl.over90_mob3.mean()k_c, n_c = int(ctrl.over90_mob3.sum()), len(ctrl)ex["trimestre"] = (ex.meses_desde_saida // 3).astype(int)linhas = []for tr, g in ex.groupby("trimestre"):    n, k = len(g), int(g.over90_mob3.sum())    lo, hi = wilson(k, n)    d, dlo, dhi = ic_diferenca(k, n, k_c, n_c)    linhas.append(dict(trimestre=tr, meses=f"{tr*3}-{tr*3+2}", n=n,                       taxa=k / n, ic_baixo=lo, ic_alto=hi,                       dif=d, dif_baixo=dlo, dif_alto=dhi,                       convergiu=bool(dlo <= 0 <= dhi)))cur = pd.DataFrame(linhas)# Convergencia = primeiro trimestre em que o IC da diferenca contem zero e# assim permanece por TRES trimestres seguidos. Tres, e nao "todos os# seguintes": as caudas tem poucos casos e uma oscilacao de um ponto so# nao deve derrubar um criterio de politica. Trimestres com menos de 200# observacoes ficam fora do teste, e aparecem no grafico em tom apagado.N_MIN, SEGUIDOS = 200, 3val = cur[cur.n >= N_MIN].reset_index(drop=True)conv = Nonefor i in range(len(val) - SEGUIDOS + 1):    if val.convergiu[i:i + SEGUIDOS].all():        conv = int(val.trimestre[i])        breakcur["testavel"] = cur.n >= N_MINcur["taxa_controle"] = taxa_ctrlcur.to_csv("curva_suavizacao.csv", index=False)print(f"coorte (regularizou o principal e hoje esta leve): {len(ex):,}"      .replace(",", "."))print("composicao por perfil de origem:")print((ex.arquetipo.value_counts(normalize=True) * 100).round(1).to_string())print(f"controle pareado: {N_CONTROLE:,}".replace(",", ".")      + f"   taxa de over90 do controle: {taxa_ctrl:.2%}")print()print(cur.assign(    taxa=lambda d: (d.taxa * 100).round(2),    dif=lambda d: (d.dif * 100).round(2),    dif_baixo=lambda d: (d.dif_baixo * 100).round(2),    dif_alto=lambda d: (d.dif_alto * 100).round(2),)[["meses", "n", "taxa", "dif", "dif_baixo", "dif_alto", "convergiu", "testavel"]]    .to_string(index=False))if conv is not None:    print(f"\nCONVERGENCIA no trimestre {conv} "          f"= {conv*3} a {conv*3+2} meses apos a baixa "          f"({SEGUIDOS} trimestres seguidos com zero dentro do IC).")    print(f"O registro negativo, porem, permanece 60 meses por lei "          f"(Lei 12.414/2011).")    print(f"Janela de penalizacao sem lastro comportamental: "          f"~{60 - (conv*3+2)} meses.")else:    print("\nNao houve convergencia sustentada dentro de 36 meses.")# por arquetipo, so para mostrar que a velocidade nao e unicaprint("\nvelocidade de recuperacao difere por perfil de origem:")por = ex.groupby("arquetipo").apply(    lambda g: pd.Series({        "n": len(g),        "over90 ate 6M": g[g.meses_desde_saida < 6].over90_mob3.mean(),        "over90 apos 24M": g[g.meses_desde_saida >= 24].over90_mob3.mean(),    }), include_groups=False)print((por.assign(n=por.n.astype(int)).round(4)).to_string())por.to_csv("suavizacao_por_arquetipo.csv")

---## 7. ExportarTabelas para anexo e os parâmetros de centroide, que permitem reaplicar a classificaçãofora do Python — é o que a planilha `Grau_Severidade_Restritivos.xlsx` faz.

In [ ]:
import pandas as pdgs = pd.read_csv("pessoas_com_gs.csv")COLS = ["id_pessoa", "GS", "comprometimento", "concentracao", "indice_natureza",        "qtd_ativos", "meses_com_inclusao_12m", "escalada", "razao_baixa",        "valor_total", "renda_mensal", "idade", "cliente_banco"]# amostra: o arquivo completo passa de 25 MB, o limite de upload do GitHubgs[COLS].sample(50_000, random_state=11).sort_values("id_pessoa") \    .round(6).to_csv("restritivos_pessoas_gs.csv", index=False)res = pd.read_csv("base_restritivos.csv")amostra = gs.id_pessoa.sample(3000, random_state=11)res[res.id_pessoa.isin(amostra)].to_csv("restritivos_amostra.csv", index=False)print("restritivos_pessoas_gs.csv : 50.000 linhas (amostra de",      f"{len(gs):,}".replace(",", "."), "pessoas)")print("restritivos_amostra.csv    :",      f"{int(res.id_pessoa.isin(amostra).sum()):,}".replace(",", "."), "linhas")print("parametros_gs.json         : centroides, media e escala")

---## O que este notebook mostra, em uma fraseQue dá para segmentar uma população sem variável resposta, e depois **provar** que asegmentação serve — usando um alvo que o modelo nunca viu. E que, quando essa prova é feitacom honestidade, ela às vezes contradiz o desenho: aqui, o quinto grau não se sustentou.